In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%%capture
!pip install trl

In [3]:
from trl import CPOConfig, CPOTrainer
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm

2025-12-29 13:53:11.113575: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767016391.313110      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767016391.365849      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767016391.827444      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767016391.827489      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767016391.827492      55 computation_placer.cc:177] computation placer alr

In [4]:
MODEL_ID = 'phuc-hoang1208/finetuned-vit5base-textsplitting'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map='auto',
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/820k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/731 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/904M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

In [5]:
train = load_dataset('vohuutridung/3190-stage2-data-v2', split='train')
validation = load_dataset('vohuutridung/3190-stage2-data-v2', split='validation')

print(train)
print(validation)

README.md:   0%|          | 0.00/183 [00:00<?, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/11525 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/378 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 11525
})
Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 378
})


# Train

In [8]:
args = CPOConfig(
    output_dir='./cpo',

    max_prompt_length=256,
    max_completion_length=256,

    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,

    warmup_ratio=0.1,

    logging_steps=200,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,

    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,

    report_to='tensorboard',
)

trainer = CPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=args,
    train_dataset=train,
    eval_dataset=validation,
)

<string>:152: FutureWarning: The `CPOConfig` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.cpo import CPOConfig`. The current import path will be removed and no longer supported in TRL 0.29. For more information, see https://github.com/huggingface/trl/issues/4223.
/tmp/ipykernel_55/736937386.py:29: FutureWarning: The `CPOTrainer` is now located in `trl.experimental`. Please update your imports to `from trl.experimental.cpo import CPOTrainer`. The current import path will be removed and no longer supported in TRL 0.29. For more information, see https://github.com/huggingface/trl/issues/4223.
  trainer = CPOTrainer(


Map:   0%|          | 0/11525 [00:00<?, ? examples/s]

Map:   0%|          | 0/11525 [00:00<?, ? examples/s]

Map:   0%|          | 0/378 [00:00<?, ? examples/s]

Map:   0%|          | 0/378 [00:00<?, ? examples/s]

Map:   0%|          | 0/11525 [00:00<?, ? examples/s]

Map:   0%|          | 0/378 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [9]:
trainer.train()

Step,Training Loss,Validation Loss,Runtime,Samples Per Second,Steps Per Second,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/rejected,Logps/chosen,Logits/rejected,Logits/chosen,Nll Loss
200,1.184800,1.143980,9.963200,37.939000,1.204000,-1.133027,-1.580382,0.528846,0.447355,-15.803822,-11.330273,-19.721848,-19.921860,0.240826
400,0.989300,1.038831,9.960600,37.949000,1.205000,-1.039305,-1.731062,0.618590,0.691757,-17.310619,-10.393048,-19.178410,-19.377302,0.221018
600,0.864000,0.982074,9.964100,37.936000,1.204000,-1.012767,-1.888934,0.645833,0.876167,-18.889336,-10.127670,-19.021250,-19.225597,0.215447
800,0.828400,0.949909,9.966200,37.928000,1.204000,-1.006215,-2.004178,0.648438,0.997963,-20.041779,-10.062152,-18.948534,-19.155499,0.214089
1000,0.786600,0.936598,9.966300,37.928000,1.204000,-1.005864,-2.062354,0.653646,1.056491,-20.623541,-10.058635,-18.959148,-19.170507,0.214042


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1083, training_loss=0.9192454935991423, metrics={'train_runtime': 3538.9517, 'train_samples_per_second': 9.77, 'train_steps_per_second': 0.306, 'total_flos': 0.0, 'train_loss': 0.9192454935991423, 'epoch': 3.0})

In [10]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(HF_TOKEN)

REPO_ID = 'vohuutridung/vit5-base-split-sft-cpo'
trainer.model.push_to_hub(REPO_ID)
trainer.processing_class.push_to_hub(REPO_ID)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/vohuutridung/vit5-base-split-sft-cpo/commit/5915c69d3d2a220864ddb5a0dc60d0f526bd336c', commit_message='Upload tokenizer', commit_description='', oid='5915c69d3d2a220864ddb5a0dc60d0f526bd336c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/vohuutridung/vit5-base-split-sft-cpo', endpoint='https://huggingface.co', repo_type='model', repo_id='vohuutridung/vit5-base-split-sft-cpo'), pr_revision=None, pr_num=None)